<a href="https://colab.research.google.com/github/Amruth-U-tech/Kolam-Classification-Project/blob/main/Kolam-Classification-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings
warnings.filterwarnings('ignore')

train_dir = "/content/drive/MyDrive/kolam dataset/kolamdataset"

In [ ]:
train_datagen = ImageDataGenerator(validation_split=0.2) # Specify validation split here

val_datagen = ImageDataGenerator(validation_split=0.2) # Specify validation split here as well

train_generator = train_datagen.flow_from_directory(train_dir, # Use train_dir
                                                    target_size=(224,224),
                                                    batch_size=16,
                                                    color_mode='rgb',
                                                    class_mode='categorical',
                                                    subset='training') # Specify subset for training
validation_generator = val_datagen.flow_from_directory(train_dir, # Use train_dir
                                                       target_size=(224,224),
                                                       batch_size=16,
                                                       color_mode='rgb',
                                                       class_mode='categorical',
                                                       subset='validation') # Specify subset for validation

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ConvNeXtTiny
from tensorflow.keras.applications.convnext import preprocess_input
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, BatchNormalization,
    RandomFlip, RandomRotation, RandomZoom, RandomTranslation, Lambda, Input
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

#Base model
convnexttiny = ConvNeXtTiny(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)
convnexttiny.trainable = False

# GPU Augmentation + Preprocessing
data_aug = tf.keras.Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.2),   # 180° rotation (0.5 = 180°)
    RandomZoom(0.2),
    RandomTranslation(height_factor=0.1, width_factor=0.1) # Added RandomTranslation
])

# Full Model
modelCT = Sequential([
    Input(shape=(224,224,3)),

    data_aug,                      # runs on GPU
    Lambda(preprocess_input),      # ConvNeXt preprocessing (GPU)

    convnexttiny,

    GlobalAveragePooling2D(),

    Dense(256, activation="relu"),
    BatchNormalization(),
    Dropout(0.5),

    Dense(11, activation="softmax")
])

#Compile
modelCT.compile(
    optimizer=Adam(learning_rate=3e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

modelCT.summary()

In [ ]:
convnexttiny.trainable = False # Freeze the entire ConvNeXtTiny model initially

# Unfreeze the last 50 layers of the ConvNeXtTiny model for fine-tuning (increased from 30)
for layer in convnexttiny.layers[-50:]:
    layer.trainable = True

modelCT.compile(
    optimizer=Adam(5e-4), # Keep the current learning rate for fine-tuning
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

In [ ]:
rom tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3, # Reduce learning rate by a factor of 0.2
    patience=5, # Reduce LR if val_loss doesn't improve for 5 epochs
    min_lr=1e-6, # Minimum learning rate
    verbose=1
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',     # Monitor the validation loss
    patience=5,            # Stop if val_loss doesn't improve for 10 epochs
    restore_best_weights=True # Restore model weights from the epoch with the best value of the monitored quantity.
)

In [ ]:
modelCT.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=50, # Keep epochs high, early_stop and reduce_lr will manage convergence
    batch_size=16,
    callbacks=[early_stop, reduce_lr] # Add the learning rate scheduler callback
)